# Cash-Secured Put 回测练习
策略：Long ETF 的替代入场方式 = Short Put + 现金担保。

Spec：legs = [{option_type: 'put', strike: 95, expiry: '2026-12-18', quantity: -1}]。

最大收益=权利金；最大亏损≈(执行价-权利金)×100；盈亏平衡=执行价-权利金。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
spot, strike, premium, size, fee, capital = 100, 95, 2, 100, 1, 20000
contracts = int(capital // (strike * size))
def pnl(expiry_price):
    return (premium - np.maximum(strike-expiry_price, 0))*size*contracts - fee*contracts
print('contracts:', contracts, 'max return:', pnl(200), 'break-even:', strike-premium-fee/size)

In [ ]:
prices = np.linspace(0, 120, 500)
expired_pnl = pnl(prices)
plt.plot(prices, expired_pnl); plt.axhline(0,color='black'); plt.axvline(strike,ls='--'); plt.axvline(strike-premium,ls='--',color='orange')
plt.title('Expired P&L'); plt.xlabel('Expiry price'); plt.ylabel('P&L'); plt.grid(alpha=.25); plt.show()

In [ ]:
dates = pd.date_range('2026-01-01', periods=8, freq='W')
spot_path = pd.Series([100,98,96,94,91,89,93,97], index=dates)
bt = pd.DataFrame({'spot':spot_path, 'pnl':pnl(spot_path)})
bt['equity'] = capital + bt.pnl; bt['peak'] = bt.equity.cummax(); bt['drawdown'] = bt.equity-bt.peak
fig, ax = plt.subplots(2,1,sharex=True); bt.equity.plot(ax=ax[0],title='P&L / Equity'); bt.drawdown.plot.area(ax=ax[1],title='Drawdown',color='crimson'); plt.tight_layout(); plt.show()
bt

## 练习
1. 修改 strike、premium、capital 和 fee。
2. 比较不同价格路径的 P&L 与最大回撤。
3. 思考被指派后如何转换成 Long ETF，再接 Covered Call。